

### 1.2 -- Physical Interpretation of the Data
In this notebook, each set of current–voltage measurements is treated as a dataset associated with a single wavelength of light. From each dataset, the primary quantity we seek to determine is the stopping potential $V_0$. Once the stopping potential has been extracted for several different wavelengths, these values are related to the frequency of the incident light through Einstein’s photoelectric equation.

This analysis involves a sequence of transformations between physical quantities: wavelength is converted to frequency, frequency is related to photon energy, and the stopping potential is used to determine fundamental constants such as Planck’s constant and the work function of the cathode material. Because each step maps one set of inputs to a well-defined output, the photoelectric effect provides a natural framework for thinking about experimental analysis in terms of functions, both mathematical functions and Python functions, whose purpose is to model physical relationships rather than simply manipulate numbers. When the stopping potential is plotted as a function of the frequency of the incident light, the resulting linear relationship allows fundamental constants to be extracted from experimental data. In particular, the slope of this relationship is proportional to Planck’s constant, while the intercept is related to the work function of the cathode material.


## Problem 1
`10 points`
 - C3.2: Relate photon properties to electron emission behavior.
 - C3.3: Interpret experimental trends using physical reasoning.

a) Why is the stopping potential associated with the maximum kinetic energy of the emitted electrons rather than the average kinetic energy?

b) For a fixed wavelength of light, why does changing the applied voltage affect the measured current but not the energy of individual photons?

c) If two different wavelengths produced the same stopping potential, what would that imply about the energy of the emitted electrons?

---

## 2 — Conversions as Functions

In the photoelectric effect, the properties of light are most naturally described in terms of **wavelength**, **frequency**, and **energy**. While wavelength is the quantity selected experimentally using optical filters, the photoelectric equation is written in terms of frequency. As a result, the first step in the analysis is to convert between these representations in a consistent and transparent way.

In this part of the notebook, you will use Python functions to carry out these conversions. The goal is not simply to compute numerical values, but to understand how each function encodes a physical relationship and how the output of one function becomes the input to the next.

---

### 2.1 — Converting Wavelength to Frequency

` 1 point`

The frequency $f$ of light is related to its wavelength $\lambda$ by the speed of light $c$:

$$
f = \frac{c}{\lambda}
$$

In the laboratory, wavelengths are specified in **nanometers**, while frequency is typically expressed in **hertz (s$^{-1}$)**. A conversion function allows us to apply this relationship consistently for every wavelength used in the experiment.

In the code cell below, you will use a provided Python function that converts a wavelength in nanometers to a frequency in hertz.

In [ ]:
from scipy.constants import speed_of_light # in m/s

def wavelength_nm_to_thz(wavelength):
    """
    Convert a wavelength in nanometers to frequency in terahertz.

    Parameters
    ----------
    wavelength : float or array-like
        Wavelength of light in nanometers (nm).

    Returns
    -------
    frequency : float or array-like
        Frequency of light in terahertz (THz).
    """
    frequency = speed_of_light * 1e-3 / wavelength
    return frequency

428.27493999999996

### 2.2 — Photon Energy as a Function of Frequency

`2 points`

Once the frequency of the incident light is known, the energy of an individual photon is given by

$$
E = h f
$$

where $h$ is Planck’s constant. Although Planck’s constant will later be *determined experimentally* in this lab, it is still useful at this stage to compute photon energies in order to compare different wavelengths qualitatively.

Write a function that takes frequency in terahertz (THz) and returns photon energy in joules (J).

In [ ]:
import numpy as np
from scipy.constants import Planck # h in J/Hz

def frequency_thz_to_energy_joule(frequency_thz):
    """
    ----------
    frequency_thz : float or array-like
        Frequency in terahertz (THz).

    Returns
    -------
    energy_joule : float or array-like
        Photon energy in joules (J).
    """
    # TODO: implement using E = h f and convert THz -> Hz
    pass

# Quick checks
print(frequency_thz_to_energy_joule(822))  # order of magnitude ~ 1e-19 J
print(frequency_thz_to_energy_joule(np.array([822, 740])))

5.4466296633e-19
[5.44662966e-19 4.90329191e-19]


### 2.3 — Light Properties Table

`2 points`

In the rest of the notebook you will repeatedly need wavelength, frequency, and photon energy for the same set of filters. Rather than recomputing these values manually each time, write a function that takes a list/array of wavelengths (nm) and returns a table containing:

- wavelength (nm)
- frequency (THz)
- photon energy (J)

Your function should call your functions from Parts 2.1 and 2.2.

In [ ]:
import pandas as pd

def make_light_properties_table(wavelengths_nm):
    """
    Build a table of wavelength (nm), frequency (THz), and photon energy (J).

    Parameters
    ----------
    wavelengths_nm : array-like
        Wavelengths in nanometers (nm).

    Returns
    -------
    table : pandas.DataFrame
        Columns: wavelength_nm, frequency_thz, energy_joule
    """
    wavelengths_nm = np.array(wavelengths_nm)
    
    # TODO: compute frequency_thz using wavelength_nm_to_thz
    # TODO: compute energy_joule using frequency_thz_to_energy_joule
    # TODO: return a DataFrame with the specified columns

    pass

filters_nm = [365, 405, 436, 546, 577]
make_light_properties_table(filters_nm)

## Problem 3

`10 points`

a) Suppose the work function of calcium is $ \phi $. Using only the photon energies computed in Part 2, determine which of the listed wavelengths are *capable* of ejecting electrons (i.e., satisfy $ E > \phi $). Describe how this prediction will be tested experimentally in Part 3.

b) In the linear photoelectric equation

$$
   eV_0 = hf - \phi,
$$

explain why frequency, rather than wavelength, is the natural independent variable for a linear model. What mathematical complication would arise if you attempted to plot $V_0$ directly as a function of wavelength?

# (NEW PART 1, 3B)


### 3B — Extracting the Stopping Potential

Write a function that takes one dataset and returns the stopping potential \(V_0\).

Conceptually, this function performs the mapping
$$
\{(V, I)\}_{\lambda} \mapsto V_0(\lambda).
$$

A standard approach is to estimate $V_0$ by fitting the **approximately linear region near the cutoff** (where the current approaches zero) and extrapolating to the x-intercept. One reasonable workflow is:

1. **Select a fitting region.** For example, keep only points with current above a small threshold $r$ to avoid the flat/noisy region very near zero:
   - Create a mask such as `mask = (I > r)`.
   - Then use `V_fit = V[mask]` and `I_fit = I[mask]`.

2. **Fit a line** to the selected points:
   $$
   I = mV + b.
   $$
   You may use `numpy.polyfit(V_fit, I_fit, deg=1)` to obtain $m$ and $b$.

3. **Solve for the stopping potential** by setting $I=0$:
   $$
   0 = mV_0 + b \quad \Rightarrow \quad V_0 = -\frac{b}{m}.
   $$

Apply your function to `Ca_200nm` and record the value of \(V_0\). Briefly justify your choice of fitting region (for example, by stating your threshold \(r\) or describing which portion of the curve you used).

### Part 4.2 — Compute $V_0$ for Each Calcium Dataset

`5 points`

Use your stopping-potential function from Part 3 to compute $V_0$ for each dataset.

You must choose a threshold value used to define the fitting region. Record your choice of threshold (including units) and use the same value for all four wavelengths unless you justify doing otherwise.

<br></br>

In [ ]:
threshold = None  # TODO: choose a threshold value appropriate for your current units

import os
# TODO: use os.listdir to get a list of data files
files = None 

V0_values = []
for file in files:
    path = './data/' + file
    # TODO: use pandas to read csv into a DataFrame 

    # TODO: get V0 and append to V0_values
    V0 = extract_stopping_potential(df, threshold=threshold)

V0_values


### Part 4.3 — Build the Calcium Results Table

`5 points`

Create a final results table by adding your stopping potentials to the light-properties table.

Your final table must include the columns:
- `wavelength_nm`
- `frequency_thz` (or `frequency_hz`, depending on how you stored it)
- `energy_joule`
- `V0_volt`

This table will be used directly in Part 5.

In [ ]:
results = light_table.copy()

# TODO: add a column named 'V0_volt' using V0_values (make sure the order matches wavelengths_nm)
results["V0_volt"] = None

results


NameError: name 'light_table' is not defined

## Problem 5

`5 points`

Answer in complete sentences.

a) As wavelength increases from 200 nm to 350 nm, how do the photon energy and stopping potential change? State whether each increases or decreases and justify using the relationships in Parts 2–3.

b) Does the trend between stopping potential and frequency appear consistent with a linear model? Explain what you would look for in the plot to support or refute linearity.

c) Which wavelength appears closest to the cutoff condition (where $V_0 \approx 0$)? Explain what that implies about the work function of the metal.

d) If one wavelength gives an unexpectedly large or small $V_0$, give one plausible experimental cause and one plausible analysis (data-processing) cause.


# temp stuff (my work on the new version)

### 1C - Filtering Tabular Data with `pandas`
Okay, that looks pretty terrible! Clearly this line of best fit cannot be used to get the stopping potential from our dataset. We will need to *filter* our data so that we are only using values from the linear region. 

This will require filtering out data from both the zero current and saturation current regions. Fortunately for us, `pandas` makes this very easy to do. An example is provided for filtering out the saturation current region; **your task** will be to filter out the zero current region and re-create the plot.
sat_current = max(ca_data['current_uA'])
# comparision operators on arrays return arrays of true/false values for filtering
sat_filter = ca_data['current_uA'] < (sat_current - 0.5) # add small threshold

# your turn!
# define the zero filter here. Don't use == 0; use a small threshold.
zero_current = # use min() here (or just use 0!)
zero_filter = # check where current is greater than zero_current + some threshold

#since these are arrays of true/ false values, we can combine them with '&'
# to find where both are true. 
total_filter = sat_filter & zero_filter

# now we can filter our data!
x_filter = ca_data['voltage_V'][total_filter]
y_filter = # filter the current data with the same filter! 